# Act 4: a planner and its harness

A planning agent assesses a confirmed menu, tracks tasks, consults vendor evidence, and requests review of a mock email. Compare five named features while keeping approval and policy checks mandatory.

Start with repeatable **offline fixtures**. This notebook requires a Python 3.13+ kernel; Colab runtime compatibility has not been validated here. Nothing is purchased or emailed. Approval saves only a local HTML artifact.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

root = Path.cwd()
if not (root / "formaggio" / "config.py").exists():
    if (root.parent / "formaggio" / "config.py").exists():
        root = root.parent
    else:
        root = root / "agent-harness"
        if not root.exists():
            subprocess.run(["git", "clone", "https://github.com/dlops-io/agent-harness.git", str(root)], check=True)
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("Repository:", root)

In [ ]:
import tomllib

if sys.version_info < (3, 13):
    raise RuntimeError("Select a Python 3.13+ notebook kernel before continuing.")
project = tomllib.loads(Path("pyproject.toml").read_text())
subprocess.run([sys.executable, "-m", "pip", "install", *project["project"]["dependencies"]], check=True)
subprocess.run([sys.executable, "-m", "scripts.check_sdk"], check=True)

## 1. Run Act 4

The fixture checks the confirmed event menu, identifies a 300 g shortfall of Époisses, and requests approval for a vendor availability inquiry. The host callback below explicitly declines it.


In [ ]:
from pathlib import Path
from acts.act4_harness import build_act4, build_planner
from cli import print_harness_result
from formaggio.operations.observability import Recorder

DB_PATH = Path("outputs/act4_notebook.sqlite")
OUTPUT_ROOT = Path("outputs/act4_notebook")

async def fixture_review(review):
    print(f"Review: {review.draft.recipient} — {review.draft.subject}")
    print(review.draft.body)
    return "decline"  # Change to approve to save the exact draft as local HTML.



In [ ]:
lesson = build_act4(fixture=True, output_root=OUTPUT_ROOT, decision_source="fixture")
with Recorder(DB_PATH) as recorder:
    result = await lesson.run(recorder, reviewer=fixture_review, progress=print)
print_harness_result(result)


## 2. Read the planner and its named features

The planner is an SDK agent with narrow tools and session history. The harness adds trace, budget, planning, memory, and compaction. A completed task never authorizes an action.


In [ ]:
import inspect
print(inspect.getsource(build_planner))
print(inspect.getsource(build_act4))


## 3. Remove planning and stored memory

**Agent harness:** removing planning removes task tools. **Context engineering:** removing memory skips its database, while preferences confirmed in the current request remain available. Each run gets fresh task, review and conversation state.


In [ ]:
minimal_lesson = lesson.without("planning").without("memory")
with Recorder(DB_PATH) as recorder:
    minimal_result = await minimal_lesson.run(recorder, reviewer=fixture_review, progress=print)
print_harness_result(minimal_result)


## 4. Separate compaction from current state

Removing compaction keeps history. It still refreshes confirmed constraints and approval state before each model call. The context size cap remains enforced.


In [ ]:
with Recorder(DB_PATH) as recorder:
    uncompacted_result = await lesson.without("compaction").run(
        recorder, reviewer=fixture_review, progress=print)
    states = [event["payload"]["capsule"] for event in recorder.timeline(uncompacted_result["run_id"])
              if event["event_type"] == "context.model_input"]
print("Confirmed allergies:", states[-1]["confirmed_request"]["allergies"])
print("Review decisions:", states[-1]["email_decisions"])


Now demonstrate compaction with explicitly synthetic old history and a malicious vendor document. The native SDK evicts older history; the application restores current state. Vendor text is quarantined and cannot grant approval.


In [ ]:
compaction_lesson = build_act4(fixture=True, demo_compaction=True, document="malicious",
                              output_root=OUTPUT_ROOT, decision_source="fixture")
with Recorder(DB_PATH) as recorder:
    compacted_result = await compaction_lesson.run(recorder, reviewer=fixture_review, progress=print)
print_harness_result(compacted_result)


## 5. Try the live planner (optional)

These cells make model API calls. Review the recipient and body before approving a local save. Human wait time does not consume the active execution budget; model and tool counts remain cumulative after review.


In [ ]:
import os
from getpass import getpass
from formaggio.config import MODEL

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")


In [ ]:
async def review_email(review):
    print(f"Recipient: {review.draft.recipient}\nSubject: {review.draft.subject}\n{review.draft.body}")
    while True:
        decision = input("Approve local HTML save or decline? ").strip().lower()
        if decision in {"approve", "decline"}:
            return decision

with Recorder(DB_PATH) as recorder:
    live_result = await build_act4(model=MODEL, output_root=OUTPUT_ROOT).run(
        recorder, reviewer=review_email, progress=print)
print_harness_result(live_result)


## Reflect

- Which state belongs to this invocation, and which preferences persist between runs?
- Why do confirmed constraints and decisions remain available without compaction?
- Which checks remain active when trace or budget is removed?
- **Evaluation:** compare the authoritative status and artifacts with the agent's prose and task list. A fixture verifies a known execution path; it does not establish live model quality.

Act 5 adds reusable skills to guide the planner's procedure and presentation.
